# 03 — Evaluate GTE-ModernBERT on CSFCube

Each query paper and candidate paper is represented as title plus abstract.
Both the pinned zero-shot checkpoint and the DORIS-MAE-adapted checkpoint
score every released candidate exactly once for each facet.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import gc
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers.cross_encoder import CrossEncoder

from tools.modernbert_supervised_baseline import (
    build_csfcube_rankings,
    read_jsonl,
    reproducibility_manifest,
    sha256_file,
    write_json,
)

ARTIFACT_DIR = Path(os.environ.get(
    "RANKING_PRETRAINED_COMPACT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "ranking_pretrained_compact_baseline"),
))
MODEL_ID = "Alibaba-NLP/gte-reranker-modernbert-base"
MODEL_REVISION = "204bdd4d405882a9e8ca9452c223fc49f02e8451"
ADAPTED_MODEL_DIR = ARTIFACT_DIR / "models" / "gte_modernbert_dorismae_lambda_seed3407" / "final"
CSFCUBE_DIR = REPO_ROOT / "CSFCube"
PAPER_PATH = CSFCUBE_DIR / "abstracts-csfcube-preds.jsonl"
OUTPUT_DIR = ARTIFACT_DIR / "csfcube"
FACETS = ("background", "method", "result")
MAX_LENGTH = 1024
PREDICT_BATCH_SIZE = 64

MODEL_VARIANTS = {
    "zero_shot": {
        "source": MODEL_ID,
        "revision": MODEL_REVISION,
        "experiment": "gte-modernbert-zeroshot",
    },
    "dorismae_adapted": {
        "source": str(ADAPTED_MODEL_DIR),
        "revision": None,
        "experiment": "gte-modernbert-dorismae-lambda-seed3407",
    },
}

if not ADAPTED_MODEL_DIR.exists():
    raise FileNotFoundError(f"Run notebook 01 first; missing {ADAPTED_MODEL_DIR}")
if not PAPER_PATH.exists():
    raise FileNotFoundError(PAPER_PATH)
if not hasattr(np, "asfarray"):
    raise RuntimeError(
        "The unchanged CSFCube rank_metrics.py requires NumPy 1.26.4; "
        f"the active kernel has NumPy {np.__version__}. Run "
        "`python -m pip install numpy==1.26.4`, restart the kernel, "
        "and rerun this notebook."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
paper_rows = read_jsonl(PAPER_PATH)
paper_lookup = {}
for item in paper_rows:
    abstract = item.get("abstract", "")
    if isinstance(abstract, list):
        abstract = " ".join(str(sentence) for sentence in abstract)
    paper_lookup[int(item["paper_id"])] = (
        f"Title: {item.get('title', '')}\nAbstract: {abstract}"
    )

print(f"Paper texts: {len(paper_lookup):,}")

In [ ]:
run_records = {}
for variant_name, variant in MODEL_VARIANTS.items():
    model_kwargs = {"max_length": MAX_LENGTH}
    if variant["revision"] is not None:
        model_kwargs["revision"] = variant["revision"]
    model = CrossEncoder(variant["source"], **model_kwargs)

    def score_pairs(pairs):
        return model.predict(
            pairs,
            batch_size=PREDICT_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
        )

    run_records[variant_name] = {}
    for facet in FACETS:
        gold_path = CSFCUBE_DIR / f"test-pid2anns-csfcube-{facet}.json"
        annotations = json.loads(gold_path.read_text(encoding="utf-8"))
        expected_sets = {
            str(query_id): {str(value) for value in item["cands"]}
            for query_id, item in annotations.items()
        }

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        started = time.perf_counter()
        rankings = build_csfcube_rankings(annotations, paper_lookup, score_pairs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - started

        assert set(rankings) == set(expected_sets)
        assert all(
            {item[0] for item in rankings[query_id]} == expected
            for query_id, expected in expected_sets.items()
        )
        output_path = OUTPUT_DIR / (
            f"test-pid2pool-csfcube-{variant['experiment']}-{facet}-ranked.json"
        )
        write_json(output_path, rankings)
        document_count = sum(len(value) for value in rankings.values())
        run_records[variant_name][facet] = {
            "gold_path": str(gold_path.resolve()),
            "gold_sha256": sha256_file(gold_path),
            "output": str(output_path.resolve()),
            "output_sha256": sha256_file(output_path),
            "queries": len(rankings),
            "documents": document_count,
            "elapsed_seconds": elapsed,
            "queries_per_second": len(rankings) / elapsed,
            "documents_per_second": document_count / elapsed,
            "peak_vram": (
                torch.cuda.max_memory_reserved() / (1024 ** 3)
                if torch.cuda.is_available() else 0.0
            ),
        }
        print(variant_name, facet, json.dumps(run_records[variant_name][facet], indent=2))

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Save evaluator stdout because ranking_eval.py reports, rather than serializes, metrics.
paired_statistics_inputs = {}
for variant_name, variant in MODEL_VARIANTS.items():
    experiment = variant["experiment"]
    for facet in (*FACETS, "all"):
        command = [
            sys.executable,
            str(CSFCUBE_DIR / "eval_scripts" / "ranking_eval.py"),
            "eval_pool_ranking",
            "--gold_path",
            str(CSFCUBE_DIR),
            "--ranked_path",
            str(OUTPUT_DIR),
            "--experiment",
            experiment,
            "--facet",
            facet,
        ]
        print("Running:", " ".join(command))
        result = subprocess.run(command, check=False, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout)
        if result.returncode != 0:
            print("CSFCube evaluator stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CSFCube evaluator failed for {variant_name}/{facet} with "
                f"exit code {result.returncode}. See stderr above."
            )
        metric_prefix = "metrics" if variant_name == "dorismae_adapted" else "metrics-zero-shot"
        (OUTPUT_DIR / f"{metric_prefix}-{facet}.txt").write_text(
            result.stdout,
            encoding="utf-8",
        )

    all_per_query_path = OUTPUT_DIR / (
        f"test-pid2pool-csfcube-{experiment}-all-perq.csv"
    )
    if not all_per_query_path.exists():
        raise FileNotFoundError(
            "CSFCube evaluation did not create the required clustered-bootstrap input: "
            f"{all_per_query_path}"
        )
    paired_statistics_inputs[variant_name] = str(all_per_query_path.resolve())
    print("Paired-statistics input:", all_per_query_path)

manifest = {
    "experiment": "gte_modernbert_zero_shot_and_dorismae_csfcube",
    "model_variants": MODEL_VARIANTS,
    "query_representation": "query-paper title plus full abstract",
    "candidate_representation": "candidate-paper title plus full abstract",
    "max_length": MAX_LENGTH,
    "predict_batch_size": PREDICT_BATCH_SIZE,
    "runs": run_records,
    "paired_statistics_inputs": paired_statistics_inputs,
    "environment": reproducibility_manifest(),
}
write_json(OUTPUT_DIR / "csfcube_evaluation_manifest.json", manifest)

Retain the three ranked JSON files and the generated `-all-perq.csv` file.
The latter contains the 50 facet cases needed by the unique-paper clustered
paired bootstrap.